# Clinical Trial Dropout Prediction

**Goal:** Predict which patients are likely to drop out of a clinical trial, and explain *why* using SHAP.

**Novelty:** Adds SHAP per-patient explanations + what-if analysis — output a real biotech team would use.

**Dataset:** Synthetic data from real-world distributions (ClinicalTrials.gov literature).

## Section 1 — Setup

In [ ]:
import sys, os
sys.path.insert(0, "../src")

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
from sklearn.model_selection import train_test_split, cross_val_score
from sklearn.preprocessing import LabelEncoder
from sklearn.linear_model import LogisticRegression
from sklearn.ensemble import RandomForestClassifier, GradientBoostingClassifier
from sklearn.metrics import classification_report, roc_auc_score, roc_curve
import xgboost as xgb
import shap
import joblib
import warnings
warnings.filterwarnings("ignore")

from generate_data import generate_trial_data
print("All imports OK")

## Section 2 — Generate Data

In [ ]:
df = generate_trial_data(n_samples=5000, random_state=42)
os.makedirs("../data", exist_ok=True)
df.to_csv("../data/clinical_trial_data.csv", index=False)
print(df.shape)
df.head()

## Section 3 — EDA

In [ ]:
print(f"Dropout rate: {df[chr(39)]dropped_out[chr(39)].mean():.1%}")
print(df.describe().round(2))

In [ ]:
fig, axes = plt.subplots(2, 3, figsize=(16,10))
fig.suptitle("Dropout Rate by Key Features", fontsize=14, fontweight="bold")

for ax, col, color, title in zip(
    [axes[0,0], axes[0,1], axes[1,2]],
    ["therapeutic_area", "trial_phase", "disease_severity"],
    ["#e74c3c", "#3498db", "#9b59b6"],
    ["Therapeutic Area", "Trial Phase", "Disease Severity"]
):
    d = df.groupby(col)["dropped_out"].mean().sort_values(ascending=False)
    ax.bar(d.index, d.values*100, color=color, alpha=0.8)
    ax.set_title(f"Dropout by {title}")
    ax.set_ylabel("Dropout Rate (%)")
    ax.tick_params(axis="x", rotation=30)

df["dist_bucket"] = pd.cut(df["distance_to_site_km"], bins=[0,10,25,50,100,300],
    labels=["<10km","10-25km","25-50km","50-100km",">100km"])
d = df.groupby("dist_bucket")["dropped_out"].mean()
axes[0,2].bar(d.index.astype(str), d.values*100, color="#2ecc71", alpha=0.8)
axes[0,2].set_title("Dropout by Distance to Site")
axes[0,2].set_ylabel("Dropout Rate (%)")

df[df["dropped_out"]==0]["age"].hist(ax=axes[1,0], alpha=0.6, label="Completed", bins=20, color="#2ecc71")
df[df["dropped_out"]==1]["age"].hist(ax=axes[1,0], alpha=0.6, label="Dropped out", bins=20, color="#e74c3c")
axes[1,0].set_title("Age Distribution")
axes[1,0].legend()

d = df.groupby("patient_reported_burden_score")["dropped_out"].mean()
axes[1,1].plot(d.index, d.values*100, marker="o", color="#e74c3c", linewidth=2)
axes[1,1].set_title("Burden Score vs Dropout")
axes[1,1].set_xlabel("Burden Score (1-10)")
axes[1,1].set_ylabel("Dropout Rate (%)")
axes[1,1].grid(True, alpha=0.3)

os.makedirs("../models", exist_ok=True)
plt.tight_layout()
plt.savefig("../models/eda_plots.png", dpi=150, bbox_inches="tight")
plt.show()
print("EDA done")

## Section 4 — Preprocessing

In [ ]:
df_model = df.drop(columns=["dist_bucket"])
cat_cols = ["sex","trial_phase","therapeutic_area","employment_status","disease_severity"]
encoders = {}
for col in cat_cols:
    le = LabelEncoder()
    df_model[col] = le.fit_transform(df_model[col])
    encoders[col] = le

X = df_model.drop(columns=["dropped_out"])
y = df_model["dropped_out"]
feature_names = X.columns.tolist()

X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, random_state=42, stratify=y)
print(f"Train: {X_train.shape}, Test: {X_test.shape}")
print(f"Features: {feature_names}")

## Section 5 — Model Comparison

In [ ]:
models = {
    "Logistic Regression": LogisticRegression(max_iter=1000, random_state=42),
    "Random Forest": RandomForestClassifier(n_estimators=100, random_state=42, n_jobs=-1),
    "Gradient Boosting": GradientBoostingClassifier(n_estimators=100, random_state=42),
    "XGBoost": xgb.XGBClassifier(n_estimators=100, random_state=42, eval_metric="logloss", verbosity=0)
}

results = {}
for name, model in models.items():
    model.fit(X_train, y_train)
    y_pred = model.predict(X_test)
    y_prob = model.predict_proba(X_test)[:,1]
    auc = roc_auc_score(y_test, y_prob)
    cv = cross_val_score(model, X_train, y_train, cv=5, scoring="roc_auc")
    results[name] = {"model":model,"auc":auc,"cv_mean":cv.mean(),"cv_std":cv.std(),"y_prob":y_prob,"y_pred":y_pred}
    print(f"{name:25s}  AUC={auc:.4f}  CV={cv.mean():.4f}+-{cv.std():.4f}")

best_name = max(results, key=lambda k: results[k]["auc"])
print(f"
Best: {best_name}")

In [ ]:
plt.figure(figsize=(8,6))
colors = ["#e74c3c","#3498db","#2ecc71","#9b59b6"]
for (name,res),color in zip(results.items(),colors):
    fpr,tpr,_ = roc_curve(y_test, res["y_prob"])
    plt.plot(fpr,tpr,label=f"{name} (AUC={res[chr(39)]auc[chr(39)]:.3f})",color=color,linewidth=2)
plt.plot([0,1],[0,1],"k--",alpha=0.4)
plt.xlabel("False Positive Rate")
plt.ylabel("True Positive Rate")
plt.title("ROC Curves")
plt.legend(loc="lower right")
plt.grid(True, alpha=0.3)
plt.tight_layout()
plt.savefig("../models/roc_curves.png", dpi=150)
plt.show()

## Section 6 — SHAP Explainability

In [ ]:
best_model = results["XGBoost"]["model"]
explainer = shap.TreeExplainer(best_model)
shap_values = explainer.shap_values(X_test)

plt.figure(figsize=(10,7))
shap.summary_plot(shap_values, X_test, feature_names=feature_names, max_display=15, show=False)
plt.tight_layout()
plt.savefig("../models/shap_summary.png", dpi=150, bbox_inches="tight")
plt.show()
print("SHAP summary saved")

In [ ]:
y_probs = results["XGBoost"]["y_prob"]
shap_exp = explainer(X_test)
highest_risk_idx = int(np.argmax(y_probs))
print(f"Highest risk patient dropout prob: {y_probs[highest_risk_idx]:.1%}")
print(f"Actual: {chr(39)}Dropped out{chr(39) if y_test.iloc[highest_risk_idx]==1 else chr(39)}Completed{chr(39)}")
shap.plots.waterfall(shap_exp[highest_risk_idx], max_display=12, show=True)

## Section 7 — What-If Analysis

In [ ]:
mid_risk_mask = (y_probs > 0.4) & (y_probs < 0.6)
mid_idx = int(np.where(mid_risk_mask)[0][0])
patient = X_test.iloc[mid_idx].copy()
print(f"Patient base risk: {y_probs[mid_idx]:.1%}")

distances = np.arange(5, 151, 5)
whatif = []
for d in distances:
    p = patient.copy()
    p["distance_to_site_km"] = d
    prob = best_model.predict_proba(pd.DataFrame([p], columns=feature_names))[:,1][0]
    whatif.append({"distance_km":d,"dropout_prob":prob})
whatif_df = pd.DataFrame(whatif)

plt.figure(figsize=(9,5))
plt.plot(whatif_df["distance_km"], whatif_df["dropout_prob"]*100, marker="o", markersize=4, color="#e74c3c", linewidth=2)
plt.axvline(patient["distance_to_site_km"], color="gray", linestyle="--", label=f"Current: {patient[chr(39)]distance_to_site_km{chr(39)]:.0f}km")
plt.axhline(50, color="orange", linestyle="--", alpha=0.7, label="50% threshold")
plt.xlabel("Distance to Site (km)")
plt.ylabel("Dropout Risk (%)")
plt.title("What-If: Distance to Site vs Dropout Risk")
plt.legend()
plt.grid(True, alpha=0.3)
plt.tight_layout()
plt.savefig("../models/whatif_distance.png", dpi=150)
plt.show()

## Section 8 — Risk Tiers & Save

In [ ]:
res_df = X_test.copy()
res_df["actual_dropout"] = y_test.values
res_df["dropout_probability"] = y_probs
res_df["risk_tier"] = pd.cut(res_df["dropout_probability"], bins=[0,0.33,0.66,1.0],
    labels=["Low Risk","Medium Risk","High Risk"])

print(res_df.groupby("risk_tier").agg(patients=("actual_dropout","count"),
    actual_dropout_rate=("actual_dropout","mean")).round(3))

joblib.dump(best_model, "../models/xgb_dropout_model.pkl")
joblib.dump(encoders, "../models/label_encoders.pkl")
joblib.dump(feature_names, "../models/feature_names.pkl")
res_df.to_csv("../models/predictions.csv", index=False)
print("All files saved!")

## Final Results Summary

| Model | Test AUC | CV AUC |
|-------|----------|--------|
| Logistic Regression | — | — |
| Random Forest | — | — |
| Gradient Boosting | — | — |
| XGBoost | — | — |

**Fill in your actual numbers above after running.**

**Top dropout risk factors (SHAP):**
1. Patient reported burden score
2. Early adverse event
3. Distance to site
4. Number of visits required
5. Trial duration